# Práctica 3: Análisis Exploratorio de Datos — Titanic

## Business Understanding

## ETAPA 1: Extracción y Exploración Inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

## ETAPA 2: Data Understanding

### Paso 1: Estructura, dimensiones y duplicados

In [ ]:
# 1. Cargar el dataset
df = pd.read_csv("titanic3.csv")
df.head()

In [ ]:
# 2. Dimensión de los datos
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.shape

In [ ]:
# 3. Estructura y tipos de datos
df.info()
print("\nTipos de datos:")
print(df.dtypes)

In [ ]:
# 4. Valores duplicados
duplicados = df.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

### Paso 2: Valores nulos y columnas a eliminar

In [ ]:
# 5. Valores nulos por columna (orden descendente)
nulos = df.isnull().sum().sort_values(ascending=False)
nulos

In [ ]:
# 6. Cantidad de nulos y porcentaje
resumen_nulos = pd.DataFrame({
    "Cantidad de nulos": df.isnull().sum(),
    "Porcentaje": round(df.isnull().sum() / len(df) * 100, 2)
}).sort_values("Cantidad de nulos", ascending=False)

resumen_nulos

#### a) Estadísticas clave del dataset

In [ ]:
df.describe(include="all")

#### b) Selección de columnas a eliminar

In [ ]:
# Columnas candidatas a eliminar por exceso de nulos o baja relevancia
columnas_eliminar = ["body", "cabin", "boat", "home.dest", "ticket"]
print("Columnas a eliminar:", columnas_eliminar)
print(resumen_nulos.loc[[c for c in columnas_eliminar if c in resumen_nulos.index]])

### Paso 3: Valores únicos por columna

In [ ]:
# 7. Valores únicos por columna
df.nunique().sort_values(ascending=False)

In [ ]:
# 8. Columnas con menos de 10 valores únicos
for columna in df.columns:
    if df[columna].nunique() < 10:
        print(f"{columna} ({df[columna].nunique()} únicos): {df[columna].unique()}")

### Paso 4: Análisis exploratorio de datos

In [ ]:
# 1. Total de pasajeros
total_pasajeros = len(df)
print(f"Total de pasajeros: {total_pasajeros}")

In [ ]:
# 2. Conteo y porcentaje de sobrevivientes / no sobrevivientes
surv_counts = df["survived"].value_counts().rename({0: "No sobrevivió", 1: "Sobrevivió"})
surv_pct = round(df["survived"].value_counts(normalize=True) * 100, 2)
surv_pct.index = surv_pct.index.map({0: "No sobrevivió", 1: "Sobrevivió"})

df_supervivencia = pd.DataFrame({
    "Conteo": surv_counts,
    "Porcentaje": surv_pct
})
df_supervivencia

In [ ]:
# 3. Conteo y porcentaje de pasajeros por sexo
df_sexo = pd.DataFrame({
    "Conteo": df["sex"].value_counts(),
    "Porcentaje": round(df["sex"].value_counts(normalize=True) * 100, 2)
})
df_sexo

In [ ]:
# 4. Sobrevivientes por sexo y porcentaje de supervivencia
surv_por_sexo = df.groupby("sex")["survived"].agg(["sum", "count", "mean"])
surv_por_sexo.columns = ["Sobrevivientes", "Total", "Tasa_supervivencia"]
surv_por_sexo["Porcentaje"] = round(surv_por_sexo["Tasa_supervivencia"] * 100, 2)
surv_por_sexo

In [ ]:
# 5. Conteo y porcentaje de pasajeros por clase
df_clase = pd.DataFrame({
    "Conteo": df["pclass"].value_counts().sort_index(),
    "Porcentaje": round(df["pclass"].value_counts(normalize=True).sort_index() * 100, 2)
})
df_clase.index.name = "Clase"
df_clase

In [ ]:
# 6. Conteo de hombres y mujeres por clase
sexo_por_clase = pd.crosstab(df["pclass"], df["sex"], margins=True, margins_name="Total")
sexo_por_clase

In [ ]:
# 7. Porcentaje de supervivencia según sibsp
surv_sibsp = df.groupby("sibsp")["survived"].agg(["count", "sum", "mean"])
surv_sibsp.columns = ["Total", "Sobrevivientes", "Tasa"]
surv_sibsp["Porcentaje"] = round(surv_sibsp["Tasa"] * 100, 2)
surv_sibsp

In [ ]:
# 8. Porcentaje de supervivencia según parch
surv_parch = df.groupby("parch")["survived"].agg(["count", "sum", "mean"])
surv_parch.columns = ["Total", "Sobrevivientes", "Tasa"]
surv_parch["Porcentaje"] = round(surv_parch["Tasa"] * 100, 2)
surv_parch

### Paso 5: Gráficas con Matplotlib

In [ ]:
# 1. Histograma general por edades
plt.figure(figsize=(10, 5))
plt.hist(df["age"].dropna(), bins=30, edgecolor="black", color="steelblue")
plt.title("Histograma general de edades de los pasajeros")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# 2. Histograma por edades: sobrevivientes vs no, por sexo
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, sexo, titulo in zip(axes, ["female", "male"], ["Mujeres", "Hombres"]):
    subset = df[df["sex"] == sexo]
    ax.hist(
        subset.loc[subset["survived"] == 0, "age"].dropna(),
        bins=20, alpha=0.6, label="No sobrevivió", color="tomato", edgecolor="black"
    )
    ax.hist(
        subset.loc[subset["survived"] == 1, "age"].dropna(),
        bins=20, alpha=0.6, label="Sobrevivió", color="seagreen", edgecolor="black"
    )
    ax.set_title(f"Edades — {titulo}")
    ax.set_xlabel("Edad")
    ax.set_ylabel("Frecuencia")
    ax.legend()

plt.suptitle("Edades: sobrevivientes vs no sobrevivientes por sexo", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Histograma por clase: edad de sobrevivientes y no sobrevivientes
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, clase in zip(axes, [1, 2, 3]):
    subset = df[df["pclass"] == clase]
    ax.hist(
        subset.loc[subset["survived"] == 0, "age"].dropna(),
        bins=15, alpha=0.6, label="No sobrevivió", color="tomato", edgecolor="black"
    )
    ax.hist(
        subset.loc[subset["survived"] == 1, "age"].dropna(),
        bins=15, alpha=0.6, label="Sobrevivió", color="seagreen", edgecolor="black"
    )
    ax.set_title(f"Clase {clase}")
    ax.set_xlabel("Edad")
    ax.set_ylabel("Frecuencia")
    ax.legend()

plt.suptitle("Edades por clase: sobrevivientes vs no sobrevivientes", y=1.02)
plt.tight_layout()
plt.show()

## ETAPA 3: Data Preparation

In [ ]:
# 1. Copia del dataframe
df_prep = df.copy()
print(f"Copia creada. Shape: {df_prep.shape}")

In [ ]:
# 2. Seleccionar columnas relevantes
columnas_relevantes = ["pclass", "survived", "name", "sex", "age", "sibsp", "parch", "fare", "embarked"]
df_prep = df_prep[columnas_relevantes]
df_prep.head()

In [ ]:
# 3. Verificar estructura
df_prep.info()
print("\nNulos actuales:")
print(df_prep.isnull().sum())

In [ ]:
# 4. sex a numérico (female -> 0, male -> 1)
df_prep["sex"] = df_prep["sex"].map({"female": 0, "male": 1})
print("Valores únicos de sex:", df_prep["sex"].unique())
df_prep[["name", "sex"]].head()

In [ ]:
# 5. Imputación: age y fare = mediana; embarked = moda
mediana_age = df_prep["age"].median()
mediana_fare = df_prep["fare"].median()
moda_embarked = df_prep["embarked"].mode()[0]

print(f"Mediana age: {mediana_age}")
print(f"Mediana fare: {mediana_fare}")
print(f"Moda embarked: {moda_embarked}")

df_prep["age"] = df_prep["age"].fillna(mediana_age)
df_prep["fare"] = df_prep["fare"].fillna(mediana_fare)
df_prep["embarked"] = df_prep["embarked"].fillna(moda_embarked)

In [ ]:
# 6. Verificar que no queden nulos
print("Nulos restantes por columna:")
print(df_prep.isnull().sum())
print(f"\nTotal de nulos: {df_prep.isnull().sum().sum()}")
df_prep.info()

In [ ]:
# Exportar dataset limpio
archivo_salida = "titanic_limpio.csv"
df_prep.to_csv(archivo_salida, index=False)
print(f"Dataset limpio exportado a: {archivo_salida}")
print(f"Shape final: {df_prep.shape}")
df_prep.head(10)

## Preguntas de negocio

In [ ]:
# 1. Interacción género × clase social
interaccion = df.groupby(["sex", "pclass"])["survived"].agg(["count", "sum", "mean"])
interaccion.columns = ["Total", "Sobrevivientes", "Tasa"]
interaccion["Porcentaje"] = round(interaccion["Tasa"] * 100, 2)
display(interaccion)

tabla = df.pivot_table(values="survived", index="pclass", columns="sex", aggfunc="mean") * 100
tabla.plot(kind="bar", figsize=(8, 5), edgecolor="black")
plt.title("Porcentaje de supervivencia: género × clase")
plt.xlabel("Clase")
plt.ylabel("% Supervivencia")
plt.xticks(rotation=0)
plt.legend(title="Sexo")
plt.tight_layout()
plt.show()

In [ ]:
# 2. Familia pequeña / viajar solo vs supervivencia
df_fam = df.copy()
df_fam["family_size"] = df_fam["sibsp"] + df_fam["parch"] + 1
df_fam["family_cat"] = pd.cut(
    df_fam["family_size"],
    bins=[0, 1, 4, 20],
    labels=["Solo", "Familia pequeña (2-4)", "Familia grande (5+)"]
)

fam_surv = df_fam.groupby("family_cat", observed=True)["survived"].agg(["count", "sum", "mean"])
fam_surv.columns = ["Total", "Sobrevivientes", "Tasa"]
fam_surv["Porcentaje"] = round(fam_surv["Tasa"] * 100, 2)
display(fam_surv)

fam_surv["Porcentaje"].plot(kind="bar", color="teal", edgecolor="black", figsize=(8, 4))
plt.title("Tasa de supervivencia según tamaño de familia")
plt.ylabel("% Supervivencia")
plt.xlabel("Categoría familiar")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Diferencia de fare entre sobrevivientes y no sobrevivientes
fare_por_surv = df.groupby("survived")["fare"].agg(["count", "mean", "median", "std"])
fare_por_surv.index = fare_por_surv.index.map({0: "No sobrevivió", 1: "Sobrevivió"})
display(fare_por_surv)

df.boxplot(column="fare", by="survived", figsize=(7, 5))
plt.title("Distribución de fare según supervivencia")
plt.suptitle("")
plt.xlabel("Survived (0=No, 1=Sí)")
plt.ylabel("Fare")
plt.tight_layout()
plt.show()

print(
    f"Diferencia de media (sobrevivió - no): "
    f"{df.loc[df['survived']==1, 'fare'].mean() - df.loc[df['survived']==0, 'fare'].mean():.2f}"
)